# Анализ аудитории витуберов

#### Импорт всего нужного

In [ ]:
import sys
sys.path.insert(1, '../util/')

from data import UserData, FollowerData
from userdata import get_userdata, get_userdata_by_login
from followers import get_followers

import numpy as np
import os
import csv
from datetime import datetime
from typing import Optional, List, Tuple, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

## Грузим фолловеров у витуберов

In [ ]:
VTUBERS_LIST_FILE_PATH = "./data/vtubers.txt"
VTUBERS = set()

with open(VTUBERS_LIST_FILE_PATH, 'r') as data_file:
    for line in data_file.readlines():
        if line.startswith("#"): # comment:
            continue
        VTUBERS.add(line.lower().strip())

In [ ]:
# Check if all vtubers exists
counter = 0
for vtuber in VTUBERS:
    counter += 1
    print(f"[{counter}/{len(VTUBERS)}] Check vtuber {vtuber}")
    assert get_userdata_by_login(vtuber) is not None

In [ ]:
print(f"Loaded {len(VTUBERS)} vtubers")

### Онлайн загрузка

In [ ]:
NUM_WORKERS = 16 # Set up for your processor and to prevent from ddos protection
FAIL_REPEATS = 5

def load_followers_execution(vtuber_login: str) -> Tuple[str, Optional[List[FollowerData]]]:
    for i in range(FAIL_REPEATS):
        result = get_followers(vtuber_login)
        if result is None:
            print("[ERROR]", "Failed loading vtuber", vtuber_login, "attempt", i)
        else:
            return vtuber_login, result
    return vtuber_login, None


# Fast load
vtuber_followers = {}
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = []

    for vtuber in VTUBERS:
        future = executor.submit(load_followers_execution, vtuber_login=vtuber)
        futures.append(future)
    
    for future in as_completed(futures):
        vtuber_login, followers_result = future.result()
        if followers_result is None:
            print(f"Cannot load `{vtuber_login}` result")
        else:
            empty_results = list(filter(lambda el: el.user is None, followers_result))
            print(f"Got `{vtuber_login}` result, empty results: {len(empty_results)}/{len(followers_result)}")
        vtuber_followers[vtuber_login] = followers_result

In [ ]:
for vtuber in VTUBERS:
    assert vtuber_followers[vtuber] is not None

### Кеширование результатов

In [ ]:
def get_cache_path():
    current_date = datetime.now()

    return os.path.join("data/", 
                        ".temp/",
                        f"followers_{current_date.year}_{current_date.month}_{current_date.day}.txt")

In [ ]:
def cache_followers_data(vtuber_followers: Dict[str, List[FollowerData]]):
    with open(get_cache_path(), 'w') as cache_file:
        csvwriter = csv.writer(cache_file, delimiter=',')
        csvwriter.writerow(["vtuber",
                            "followed_at",
                            "id",
                            "login",
                            "created_at",
                            "deleted_at",
                            "follows_count"])

        for (vtuber, value) in vtuber_followers.items():
            for follower_data in value:
                if follower_data.user is None:
                    csvwriter.writerow([vtuber,
                                        follower_data.followed_at,
                                        "",
                                        "",
                                        "",
                                        "",
                                        ""])
                else:
                    csvwriter.writerow([vtuber,
                                        follower_data.followed_at,
                                        follower_data.user.id,
                                        follower_data.user.login,
                                        str(follower_data.user.created_at),
                                        str(follower_data.user.deleted_at),
                                        follower_data.user.follows_count])

In [ ]:
cache_followers_data(vtuber_followers)

#### Загрузка закешированных результатов

In [ ]:
def load_followers_cache(filepath: str) -> Dict[str, List[FollowerData]]:
    result = {}

    with open(filepath, 'r') as cache_file:
        csvreader = csv.DictReader(cache_file, delimiter=',')

        for row in csvreader:
            vtuber = row['vtuber']
            followed_at = datetime.fromisoformat(row['followed_at'])
            if row['id']:
                id              = int(row['id'])
                login           = None if row['login'] == "None" else row['login']
                created_at      = None if row['created_at'] == "None" else datetime.fromisoformat(row['created_at'])
                deleted_at      = None if row['deleted_at'] == "None" else datetime.fromisoformat(row['deleted_at'])
                follows_count   = int(row['follows_count'])
                user_data = UserData(id=id,
                                     login=login,
                                     created_at=created_at,
                                     deleted_at=deleted_at,
                                     follows_count=follows_count)
            else:
                user_data = None
            follower_data = FollowerData(user=user_data,
                                         followed_at=followed_at)
            
            if vtuber not in result.keys():
                result[vtuber] = []
            result[vtuber].append(follower_data)
    
    return result

## Обрабатываем полученные данные

In [ ]:
data = load_followers_cache("./data/.temp/followers_2025_11_1.txt")

In [ ]:
print("vtuber,followers,existing_followers")
for (vtuber, followers) in data.items():
    print(vtuber, len(followers), len(list(filter(lambda el: el.user is not None, followers))), sep=",")

In [ ]:
# Check if data is complete
for vtuber in VTUBERS:
    assert vtuber in data.keys(), vtuber

### Анализ аудитории

In [ ]:
import matplotlib.pyplot as plt

#### Анализ подписок

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in VTUBERS:
    followers = data[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

plt.hist(auditory_data, bins="auto", log=False, cumulative=False)
plt.show() 
plt.hist(auditory_data, bins="auto", log=False, cumulative=True)
plt.show() 

Тенденция к росту, однозначно, есть.

Хотя есть два момента, которые очень инетересные, первый около 4600 дня, второй - с апреля этого года.

* Первый обусловлен следующим: `yumekomoore` и `sati_akura` резко взлетели
* Второй обусловлен следующим: `xKamysh` резко взлетела 

#### Пруф для первого момента

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in VTUBERS:
    followers = data[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

plt.hist(auditory_data, bins="auto", log=False, cumulative=False, range=(4580, 4610))
plt.show() 

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in VTUBERS:
    followers = data[vtuber]
    for follower in followers:
        auditory_data.append((vtuber, follower))

auditory_data = list(filter(lambda viewer: (viewer[1].followed_at - sunce_date).days in range(4580, 4610), auditory_data))
counters = {}
for dt in auditory_data:
    counters[dt[0]] = counters.get(dt[0], 0) + 1
print(auditory_data[100][1].followed_at)

for (k, v) in sorted(counters.items(), key=lambda el: el[1], reverse=True)[:10]:
    print(k, v)

#### Пруф для второго момента

In [ ]:
IGNORED_VTUBERS = ['xkamysh', 'qchaan_9', 'rera_seal']

sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in VTUBERS:
    if vtuber in IGNORED_VTUBERS:
        continue
    
    followers = data[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

plt.hist(auditory_data, bins="auto", log=False, cumulative=False)
plt.show() 

### Анализ роста аудитории

Счетчик уникальных фолловов (то есть не учитываются второй, третий и т.д. фолловы от одного и того же человека)

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = {}
for vtuber in VTUBERS:
    followers = data[vtuber]
    for follower in followers:
        if follower.user is not None:
            if follower.user.login not in auditory_data.keys():
                auditory_data[follower.user.login] = follower.followed_at
            auditory_data[follower.user.login] = min(auditory_data[follower.user.login], follower.followed_at)

auditory_data = auditory_data.values()
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

In [ ]:
plt.hist(auditory_data, bins="auto", log=False, cumulative=False)
plt.show() 

### Уникальность витуберов

Посмотрим, какая часть аудитории витубера смотрит только его 

In [ ]:
follows_by_user = {}
for vtuber in VTUBERS:
    for follow_data in data[vtuber]:
        if follow_data.user is not None:
            if follow_data.user.login not in follows_by_user.keys():
                follows_by_user[follow_data.user.login] = []    
            follows_by_user[follow_data.user.login].append((vtuber, follow_data))
print(len(follows_by_user.keys()))

In [ ]:
unique_users = list(filter(lambda fd: len(fd[1]) <= 1, follows_by_user.items()))
print(len(unique_users))

In [ ]:
vtuber_unique_followers_counter = {}
for (user, following) in unique_users:
    vtuber_name = following[0][0]
    vtuber_unique_followers_counter[vtuber_name] = vtuber_unique_followers_counter.get(vtuber_name, 0) + 1
for (k, v) in sorted(vtuber_unique_followers_counter.items(), key=lambda el: el[1] / len(data[el[0]]), reverse=True)[:20]:
    print(k, f"{v}/{len(data[k])}", f"{v * 100 // len(data[k])}%")

In [ ]:
plt.hist(list(map(lambda el: int(el[1] * 100 / len(data[el[0]])), vtuber_unique_followers_counter.items())), bins=50)
plt.show()

| Vtuber | Statistics | Description |
| --- | --- | --- |
| mad_demian | 3855/4519 85% | в хиатусе |
| bready_xo | 4625/5512 83% | в хиатусе |
| natakodo | 18171/22059 82% | в хиатусе |
| lastglance_ | 3371/4250 79% | в хиатусе |
| gishichi | 886/1264 70% | англо витубер??? |
| kashtan_mp4 | 2030/2982 68% | Ich weiß nicht |
| nyapuru | 8657/12733 67% | мужыыыыык |
| mukubae | 1613/2429 66% | в хиатусе |
| boopa | 8545/13370 63% | Ich weiß nicht |
| linamyth | 4732/7542 62% | в хиатусе |
| ananastya_nastya | 6511/10562 61% | Лига легендер |
| amikirisan | 5176/8408 61% | Ich weiß nicht |
| piwochan | 3942/6485 60% | Ich weiß nicht |
| naxajlka | 2142/3588 59% | Ich weiß nicht |
| lightfoxmanga | 4903/8276 59% | мужыыыыык |
| quinella_admin | 8864/15398 57% | Ich weiß nicht |
| vernirra | 10568/18474 57% | Ich weiß nicht |
| planyach | 26256/46423 56% | Ich weiß nicht |
| nekisekai | 7131/12849 55% | Ich weiß nicht |
| nnstreamercha | 2546/4724 53% | мужыыыыык |
| mamilokuchii | 8751/17174 50% | Лигалегендер, любимый у дедпи |
| yumekomoore | 15773/31344 50% | Ich weiß nicht |

In [ ]:
import plotly.express as px
import pandas as pd

uniqueness_data = np.array(list(vtuber_unique_followers_counter.items()))
follower_counts_data = np.array([len(data[vtuber]) for vtuber in uniqueness_data[:, 0]])
unique_followers_count_data = uniqueness_data[:, 1].astype("int")
unique_followers_count_percentage_data = unique_followers_count_data * 100 / follower_counts_data
df = pd.DataFrame({
    "x": follower_counts_data,
    "y": unique_followers_count_percentage_data,
    "vtuber": uniqueness_data[:, 0]
})
fig = px.scatter(df, x="x", y="y", hover_name="vtuber")
fig.show()

### Кластеризация витуберов

#### Векторизация данных

In [ ]:
VTUBER_EMBEDDINGS_SIZE = len(follows_by_user.keys())
VTUBER_EMBEDDINGS_INDEX_MAP = {}
for (index, embedding) in zip(range(VTUBER_EMBEDDINGS_SIZE), list(follows_by_user.keys())):
    VTUBER_EMBEDDINGS_INDEX_MAP[embedding] = index


def vtuber2vec(vtuber: str) -> np.ndarray:
    result = np.zeros(VTUBER_EMBEDDINGS_SIZE)
    for follower_data in data[vtuber]:
        if follower_data.user is not None:
            result[VTUBER_EMBEDDINGS_INDEX_MAP[follower_data.user.login]] = 1.0
    
    return result

In [ ]:
vtuber_vectors = {}
for vtuber in VTUBERS:
    vtuber_vectors[vtuber] = vtuber2vec(vtuber)

dataset = []
for vtuber in VTUBERS:
    dataset.append(vtuber_vectors[vtuber])
dataset = np.array(dataset)

#### Кластеризация и отрисвка

In [ ]:
import plotly.express as px
import pandas as pd
import sklearn

##### Косинусное расстояние

In [ ]:
labels = sklearn.cluster.HDBSCAN(min_cluster_size=2, max_cluster_size=100, metric="cosine").fit_predict(X=dataset)

In [ ]:
projection = sklearn.manifold.TSNE(metric="cosine", random_state=123).fit_transform(dataset)

In [ ]:
data_frame = pd.DataFrame({
    "x": projection[:, 0],
    "y": projection[:, 1],
    "Кластер": labels,
    "Имя": list(VTUBERS)
})
data_frame["Кластер"] = data_frame["Кластер"].astype(str) #convert to string

In [ ]:
fig = px.scatter(data_frame, 
                 x="x", 
                 y="y",
                 color="Кластер", 
                 hover_name="Имя")
fig.show()

### Анализ количества фолловов у зрителей

#### Количество глобальных фолловов

In [ ]:
unique_users = []
used_logins = set()
for followers in data.values():
    for follower_data in followers:
        if follower_data.user is not None and follower_data.user.login not in used_logins:
            used_logins.add(follower_data.user.login)
            unique_users.append(follower_data.user)

In [ ]:
draw_data = unique_users
draw_data = list(map(lambda el: el.follows_count, draw_data))
draw_data = list(filter(lambda el: el < 100, draw_data))
df = pd.DataFrame({
    "follows_count": draw_data
})
fig = px.histogram(df, x="follows_count")
fig.show()

#### Количество локальных фолловов

In [ ]:
vtuber_following_by_user_login = {}
for (vtuber, followers) in data.items():
    for follower_data in followers:
        if follower_data.user is not None:
            if follower_data.user.login not in vtuber_following_by_user_login.keys():
                vtuber_following_by_user_login[follower_data.user.login] = []
            vtuber_following_by_user_login[follower_data.user.login].append(vtuber)

In [ ]:
draw_data = vtuber_following_by_user_login.values()
draw_data = list(map(lambda el: len(el), draw_data))
draw_data = list(filter(lambda el: el > 0, draw_data))
df = pd.DataFrame({
    "follows_count": draw_data
})
fig = px.histogram(df, x="follows_count")
fig.show()

#### Количество абсолютных уникалов среди фолловеров

In [ ]:
filtered_data = unique_users
filtered_data = list(filter(lambda el: el.follows_count <= 1, filtered_data))
len(filtered_data)

In [ ]:
counter = {}
for user in filtered_data:
    for vtuber in vtuber_following_by_user_login[user.login]:
        counter[vtuber] = counter.get(vtuber, 0) + 1
absolute_unique_followers = []
follows_count = []
vtuber_names = []
for (vtuber_name, vtuber_abs_uniq_followers) in counter.items():
    absolute_unique_followers.append(vtuber_abs_uniq_followers)
    follows_count.append(len(data[vtuber_name]))
    vtuber_names.append(vtuber_name)
absolute_unique_followers = np.array(absolute_unique_followers)
follows_count = np.array(follows_count)
vtuber_names = np.array(vtuber_names)
df = pd.DataFrame({
    "absolute_unique_followers": absolute_unique_followers / follows_count * 100,
    "follows_count": follows_count,
    "vtuber_name": vtuber_names,
})
fig = px.scatter(df, x="follows_count", y="absolute_unique_followers", hover_name="vtuber_name")
fig.show()